# Smart-Money Behavior Fingerprint MTL

This notebook tests a specific trading hypothesis:

> If smart-money groups and individual actors have different realized trading performance, and if those actors repeatedly trade in similar issuer states, then their behavior should leave learnable fingerprints in event-date feature-family data. Those fingerprints may improve future trading models by helping separate high-quality smart-money behavior from weaker or noisier behavior.

The goal is not to copy public disclosures after they are filed. For congressional and insider trades, the economically interesting date is the transaction/event date: the day the actor chose to trade. The model should learn the issuer state that existed when the behavior happened, not a delayed disclosure-following strategy.

Prior EDA motivates this notebook: congress vs senate, individual politicians, analyst actors/firms, and corporate insider roles such as CEO, CFO, director, and officer may have different forward-return profiles. If those performance differences are real, the next question is whether the behavior that produced them is learnable from the data available around the event.

The experiment has four connected layers:

1. Event direction: can issuer feature families distinguish mirrored actions such as congressional buy vs sell, insider buy vs sell, analyst upgrade vs downgrade, price-target raise vs cut, and earnings beat vs miss?
2. Actor style: can the model identify individual actors or firms from the feature states they tend to trade, such as congressional traders, analyst actors/firms, and corporate insiders?
3. Role style: can broader groups be identified, such as House vs Senate or CEO vs CFO/director/officer? This connects directly to earlier performance comparisons by actor group.
4. Profile context: use the existing `fmp_equity_profile` feature family for known issuer context such as sector, industry, exchange, and country so actor/event tasks can learn where a behavior has and has not been observed.

A useful result does not require directly trading the identity predictions. The identity/style tasks can be auxiliary multitask objectives. If they force the transformer to learn behaviorally meaningful representations, they may improve financial tasks such as trade direction, oracle-trade labels, forward-return buckets, or later ranking models.

Notebook shape:

- Universe: FMP equities with `market_cap >= 1_000_000_000_000` for this smoke-scale run.
- Timing: event/transaction dates, not delayed disclosure dates.
- Rows: one row per actual event and covered `(feature_family, target_task)` pair.
- Text: feature-family key-value pairs only. Target labels such as actor names, actor roles, and event side are not copied into text. Issuer profile context such as symbol, company name, sector, industry, exchange, and country is included through the existing `fmp_equity_profile` feature family.
- Targets: mirrored event-pair tasks plus actor/style tasks. Profile fields are features, not prediction targets. Oracle labels are intentionally excluded from this notebook.

Interpretation: this 1T+ run is primarily a pipeline and signal smoke test. Weak actor results here do not prove smart-money behavior is not learnable; they usually mean the mega-cap universe is too narrow or actor labels are too sparse. The more important scaling question is whether actor/role tasks and financial outcomes improve as we add more symbols and more repeated events.

In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
import math
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

WAREHOUSE_REPO = REPO_ROOT.parent / "quant-warehouse"
if WAREHOUSE_REPO.exists() and str(WAREHOUSE_REPO) not in sys.path:
    sys.path.insert(0, str(WAREHOUSE_REPO))

warnings.filterwarnings("ignore", category=UserWarning)

import torch
import flair
from flair.data import Dictionary, Sentence
from flair.embeddings import TransformerDocumentEmbeddings
from flair.models import MultitaskModel, TextClassifier
from flair.nn.decoder import DeepNCMDecoder
from flair.trainers import ModelTrainer

from quant_orchestrator.platforms.ml_frameworks.flair.data_adapter import (
    FlairMultitaskColumns,
    build_multitask_corpus,
)
from quant_orchestrator.platforms.ml_frameworks.flair.runtime import configure_flair_runtime

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EVENT_PAIR_TAXONOMY, EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    EventFeatureDatasetConfig,
    FamilyEvaluationConfig,
    build_event_context,
    build_event_feature_text_dataset,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_identity_text_dataset,
    cap_features_by_quality,
    evaluate_feature_target_matrix,
    event_pair_task_specs,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

In [ ]:
RANDOM_SEED = 20260701
MIN_MARKET_CAP = 100_000_000_000
START_DATE = "1900-01-01"
END_DATE = None
TRAIN_END = pd.Timestamp("2023-12-31")
DEV_END = pd.Timestamp("2024-12-31")

MIN_ROWS_PER_TASK = 120
MIN_POSITIVE_ROWS_PER_TASK = 10
MIN_FEATURE_COVERAGE = 0.50
MIN_IDENTITY_LABEL_ROWS = 10
MAX_IDENTITY_LABELS = 100
MAX_ROWS_PER_TASK_SPLIT = None
SMOKE_TEST = False
SMOKE_SYMBOL_LIMIT = 4
SMOKE_FEATURE_FAMILY_LIMIT = None

REQUIRE_CUDA = True
FLAIR_TRANSFORMER = "prajjwal1/bert-tiny"
FLAIR_EPOCHS = 1
FLAIR_BATCH_SIZE = 128
FLAIR_BATCH_CHUNK_SIZE = 8
FLAIR_EVAL_BATCH_SIZE = 64
FLAIR_LEARNING_RATE = 1e-4
FINE_TUNE_TRANSFORMER = False
DEEPNCM_USE_ENCODER = True
DEEPNCM_ENCODING_DIM = None
DEEPNCM_MEAN_UPDATE_METHOD = "online"

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebooks" / "mult-ml-frameworks" / "flair_feature_family_identity_mtl"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE_INFO = configure_flair_runtime(require_cuda=REQUIRE_CUDA)
TORCH_DEVICE = torch.device(DEVICE_INFO.torch_device)
print(DEVICE_INFO)

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    horizons=(20, 60),
    min_observations=MIN_ROWS_PER_TASK,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider="fmp",
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        "congress",
        "insider",
        "analyst_rating",
        "price_target",
        "earnings",
    ),
    oracle_trade_k_by_frequency=None,
)

runtime_config = pd.DataFrame(
    [
        {"setting": "min_market_cap", "value": f"{MIN_MARKET_CAP:,}"},
        {"setting": "date_range", "value": f"{START_DATE} to {END_DATE or 'latest warehouse row'}"},
        {"setting": "event_families", "value": ", ".join(TARGET_CONFIG.event_families)},
        {"setting": "oracle_labels", "value": "excluded"},
        {"setting": "identity_tasks", "value": "event rows only: symbol, year, sector, industry, exchange, congress actor, analyst actor, insider actor, congress chamber, insider role"},
        {"setting": "min_identity_label_rows", "value": MIN_IDENTITY_LABEL_ROWS},
        {"setting": "max_identity_labels", "value": MAX_IDENTITY_LABELS},
        {"setting": "deepncm", "value": f"use_encoder={DEEPNCM_USE_ENCODER}, method={DEEPNCM_MEAN_UPDATE_METHOD}"},
        {"setting": "torch_device", "value": str(TORCH_DEVICE)},
        {"setting": "flair_transformer", "value": FLAIR_TRANSFORMER},
    ]
)
display(runtime_config)

## Load 1T+ FMP Universe and Event Data

The universe is intentionally small here so the full MTL pipeline can be validated quickly. The event data is used in two ways:

- mirrored behavior labels, such as buy vs sell or raise vs cut;
- actor/style labels, such as congressional trader, analyst actor/firm, corporate insider, chamber, and insider role.

No oracle labels are built in this notebook. The focus is whether event-date feature-family context contains learnable smart-money behavior and identity/style structure.

In [ ]:
started = perf_counter()
WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(backend=WAREHOUSE.backend, catalog=WAREHOUSE.catalog)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)
if SMOKE_TEST:
    symbols = symbols[:SMOKE_SYMBOL_LIMIT]

print(
    {
        "symbols": len(symbols),
        "sample": symbols[:10],
        "universe_source": universe_source,
        "screen_seconds": round(perf_counter() - started, 3),
    }
)
display(universe_eligibility["reason"].value_counts().rename_axis("reason").reset_index(name="rows"))

In [ ]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics.loc[event_diagnostics["combined_rows"].gt(0), "symbol"].sort_values().tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel["symbol"].isin(event_symbols)].copy()

selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    raw_feature_metadata,
    max_features=int(raw_feature_metadata["feature"].nunique()),
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[["symbol", "date"]],
    events,
    TARGET_CONFIG,
)

target_summary = summarize_binary_targets(event_target_panel, event_target_metadata)
feature_inventory = (
    selected_feature_metadata
    .groupby(["source", "family"])
    .agg(feature_count=("feature", "nunique"))
    .reset_index()
    .sort_values(["source", "family"])
)

print(
    {
        "raw_feature_panel_rows": len(raw_feature_panel),
        "event_symbols": len(event_symbols),
        "event_rows": len(events),
        "feature_panel_rows": len(feature_panel),
        "selected_features": len(selected_features),
        "feature_families": selected_feature_metadata["family"].nunique(),
        "event_target_columns": event_target_metadata["target"].nunique(),
        "event_load_seconds": round(event_load_seconds, 3),
    }
)
display(feature_diagnostics.sort_values(["status", "rows"], ascending=[True, False]).head(30))
display(feature_inventory)
display(target_summary.sort_values(["target_family", "positive_rows"], ascending=[True, False]).head(80))

## Actor Behavior Coverage

These tables inspect whether the current universe has enough repeated actor metadata to study behavior.

This matters because the economic premise comes from performance differences across groups and individuals. If senators, House members, CEOs, CFOs, directors, analyst firms, or specific actors have different realized performance, then a model benefits only if those groups also have learnable behavioral fingerprints.

The tables below are descriptive EDA before modeling. Actor names, analyst firms, congress chamber, and insider role are labels or grouping keys, not text features.

In [ ]:
def normalize_congress_chamber(value: object) -> str | None:
    text = str(value).strip().lower() if pd.notna(value) else ""
    if not text:
        return None
    if "senate" in text or "senator" in text:
        return "senate"
    if "house" in text or "representative" in text or "rep." in text:
        return "house"
    return "congress_other"


def normalize_insider_role(value: object) -> str | None:
    text = str(value).strip().lower() if pd.notna(value) else ""
    if not text:
        return None
    if "chief executive" in text or re.search(r"\bceo\b", text):
        return "ceo"
    if "chief financial" in text or re.search(r"\bcfo\b", text):
        return "cfo"
    if "director" in text:
        return "director"
    if "president" in text:
        return "president"
    if "officer" in text:
        return "officer"
    if "10%" in text or "ten percent" in text or "beneficial owner" in text:
        return "large_holder"
    return "other_insider"


def clean_label(value: object) -> str | None:
    text = str(value).strip() if pd.notna(value) else ""
    return text if text else None


def coalesce_label(frame: pd.DataFrame, primary: str, fallback: pd.Series | None = None) -> pd.Series:
    values = frame[primary].map(clean_label) if primary in frame.columns else pd.Series([None] * len(frame), index=frame.index)
    if fallback is not None:
        values = values.where(values.notna(), fallback.map(clean_label))
    return values


def enrich_actor_style_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    event_family = out["event_family"].astype(str) if "event_family" in out.columns else pd.Series("", index=out.index)
    actor_type = out["actor_type"] if "actor_type" in out.columns else pd.Series([None] * len(out), index=out.index)
    actor_name = out["actor_name"] if "actor_name" in out.columns else pd.Series([None] * len(out), index=out.index)

    out["congress_chamber"] = np.where(
        event_family.eq("congress"),
        coalesce_label(out, "actor_chamber", actor_type.map(normalize_congress_chamber)),
        None,
    )
    out["insider_role"] = np.where(
        event_family.eq("insider"),
        coalesce_label(out, "actor_role", actor_type.map(normalize_insider_role)),
        None,
    )
    out["analyst_actor"] = np.where(
        event_family.isin(["analyst_rating", "price_target"]),
        coalesce_label(out, "actor_firm", actor_name),
        None,
    )
    return out


def event_actor_eda(events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frame = enrich_actor_style_columns(events)
    frame["event_date"] = pd.to_datetime(frame["event_date"], errors="coerce").dt.normalize()
    frame["event_year"] = frame["event_date"].dt.year

    family_summary = (
        frame.groupby("event_family")
        .agg(
            rows=("event_type", "size"),
            symbols=("symbol", "nunique"),
            actors=("actor_name", "nunique"),
            actor_types=("actor_type", "nunique"),
            actor_roles=("actor_role", "nunique") if "actor_role" in frame.columns else ("actor_name", "nunique"),
            actor_firms=("actor_firm", "nunique") if "actor_firm" in frame.columns else ("actor_name", "nunique"),
            min_date=("event_date", "min"),
            max_date=("event_date", "max"),
        )
        .reset_index()
        .sort_values("rows", ascending=False)
    )

    side_by_actor_type = (
        frame.groupby(["event_family", "actor_type", "event_type"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["event_family", "rows"], ascending=[True, False])
    )

    role_summary = pd.concat(
        [
            frame.loc[frame["congress_chamber"].notna()]
            .groupby(["congress_chamber", "event_type"])
            .size()
            .rename("rows")
            .reset_index()
            .assign(group="congress_chamber")
            .rename(columns={"congress_chamber": "label"}),
            frame.loc[frame["insider_role"].notna()]
            .groupby(["insider_role", "event_type"])
            .size()
            .rename("rows")
            .reset_index()
            .assign(group="insider_role")
            .rename(columns={"insider_role": "label"}),
        ],
        ignore_index=True,
    ).sort_values(["group", "rows"], ascending=[True, False])

    top_actors = (
        frame.loc[frame["actor_name"].notna() & frame["actor_name"].astype(str).str.strip().ne("")]
        .groupby(["event_family", "actor_name", "event_type"])
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["event_family", "rows"], ascending=[True, False])
        .groupby("event_family")
        .head(20)
        .reset_index(drop=True)
    )
    return family_summary, side_by_actor_type, role_summary, top_actors


actor_family_summary, actor_type_summary, actor_role_summary, top_actor_summary = event_actor_eda(events)
print({"event_columns": list(events.columns), "event_rows": len(events)})
display(actor_family_summary)
display(actor_type_summary.head(80))
display(actor_role_summary.head(80))
display(top_actor_summary.head(100))

## Select Usable Event Tasks and Feature Families

This step finds feature-family/target pairs with enough event coverage for modeling.

The selected feature families are reused across event-direction, actor-style, role-style, and regime-context tasks. This keeps the experiment aligned with the eventual transformer shape: one row is `(date, symbol, feature_family, target_task)` with the feature family rendered as key-value text.

In [ ]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    selected_feature_metadata,
    event_target_panel,
    event_target_metadata,
    min_rows=MIN_ROWS_PER_TASK,
    min_positive_rows=MIN_POSITIVE_ROWS_PER_TASK,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
)

usable_matrix = matrix.query("status == 'usable'").copy()
if SMOKE_TEST and SMOKE_FEATURE_FAMILY_LIMIT is not None and not usable_matrix.empty:
    keep_families = (
        usable_matrix
        .sort_values(["mean_abs_smd", "positive_rows"], ascending=[False, False])[["source", "feature_family"]]
        .drop_duplicates()
        .head(int(SMOKE_FEATURE_FAMILY_LIMIT))
    )
    usable_matrix = usable_matrix.merge(keep_families, on=["source", "feature_family"], how="inner")

usable_targets = set(usable_matrix["target"].astype(str))
usable_feature_families = usable_matrix[["source", "feature_family"]].drop_duplicates().sort_values(["source", "feature_family"])

print(
    {
        "matrix_rows": len(matrix),
        "usable_pairs": len(usable_matrix),
        "training_panel_rows": len(training_panel),
        "usable_event_targets": len(usable_targets),
        "usable_feature_families": len(usable_feature_families),
    }
)
display(usable_matrix.head(80))

## Text and Task Builders

The core dataset invariant is event-first:

- Start from actual event rows only.
- Inner join covered feature-family rows on the same `(symbol, date)`.
- Never create feature-family training rows for dates where the event did not happen.
- Never use non-events as negative examples for event-pair tasks.

Profile fields are handled as features, not identity targets. The notebook reuses the existing `fmp_equity_profile` feature family for known issuer context such as country, sector, industry, and exchange. This lets the shared MTL representation learn familiarity boundaries like “Nancy-like setup on Mag7” versus “same setup outside the observed regime.”

Leakage guardrails:

- `symbol`, `date`, `year`, and `year_label` are not included in feature-family text.
- Actor names, analyst actors, chamber, insider role, and other actor metadata are labels only and are not copied into text.
- Actor/style tasks use only actual matching event rows: congress actors from congress events, analyst actors from analyst/price-target events, and insider actors/roles from insider events.

This design tests whether actor and role fingerprints are present in issuer feature states plus known profile context, not whether labels were accidentally leaked into the text.

In [ ]:
DATASET_CONFIG = EventFeatureDatasetConfig(
    min_feature_coverage=MIN_FEATURE_COVERAGE,
    min_label_rows=MIN_IDENTITY_LABEL_ROWS,
    max_labels=MAX_IDENTITY_LABELS,
)

PROFILE_FEATURE_SOURCE = "fmp"
PROFILE_FEATURE_FAMILY = "fmp_equity_profile"
PROFILE_FEATURE_COLUMNS = ["date", "symbol", "company_name", "exchange", "country", "sector", "industry"]
PROFILE_TEXT_FEATURE_COLUMNS = ["symbol", "company_name", "exchange", "country", "sector", "industry"]


def add_profile_feature_family(panel: pd.DataFrame, metadata: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    profile_rows = []
    for symbol in panel["symbol"].astype(str).str.upper().drop_duplicates().sort_values():
        profile = WAREHOUSE.catalog.get_profile(symbol=symbol, provider=FEATURE_CONFIG.provider)
        profile_rows.append(
            {
                "symbol": symbol,
                "company_name": profile.company_name if profile is not None else None,
                "exchange": profile.exchange if profile is not None else None,
                "country": profile.country if profile is not None else None,
                "sector": profile.sector if profile is not None else None,
                "industry": profile.industry if profile is not None else None,
            }
        )
    profile_frame = pd.DataFrame(profile_rows)
    out_panel = panel.merge(profile_frame, on="symbol", how="left", suffixes=("", "_profile"))
    for column in PROFILE_FEATURE_COLUMNS:
        fallback = f"{column}_profile"
        if fallback in out_panel.columns:
            out_panel[column] = out_panel[column].where(out_panel[column].notna(), out_panel[fallback]) if column in out_panel.columns else out_panel[fallback]
            out_panel = out_panel.drop(columns=[fallback])
    profile_metadata = pd.DataFrame(
        [
            {
                "feature": column,
                "family": PROFILE_FEATURE_FAMILY,
                "source": PROFILE_FEATURE_SOURCE,
                "source_column": column,
                "expected_direction": "categorical",
            }
            for column in PROFILE_TEXT_FEATURE_COLUMNS
        ]
    )
    out_metadata = (
        pd.concat([metadata, profile_metadata], ignore_index=True)
        .drop_duplicates(["source", "family", "feature"])
        .sort_values(["family", "feature"])
        .reset_index(drop=True)
    )
    return out_panel, out_metadata


def allowed_feature_families() -> set[tuple[str, str]] | None:
    if SMOKE_TEST and SMOKE_FEATURE_FAMILY_LIMIT is not None:
        allowed = set(map(tuple, usable_feature_families[["source", "feature_family"]].to_numpy()))
        allowed.add((PROFILE_FEATURE_SOURCE, PROFILE_FEATURE_FAMILY))
        return allowed
    return None


def event_task_specs() -> list[dict[str, object]]:
    return event_pair_task_specs(TARGET_CONFIG, usable_targets)


ALLOWED_FEATURE_FAMILIES = allowed_feature_families()
print({"usable_feature_families": len(usable_feature_families), "event_task_specs": len(event_task_specs()), "profile_feature_family": PROFILE_FEATURE_FAMILY})

ACTOR_STYLE_COLUMNS = ["analyst_actor", "congress_chamber", "insider_role"]

In [ ]:
dataset_started = perf_counter()
model_panel, model_feature_metadata = add_profile_feature_family(training_panel, selected_feature_metadata)

event_result = build_event_feature_text_dataset(
    model_panel,
    model_feature_metadata,
    event_task_specs(),
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "event_feature_text_dataset", "rows": len(event_result.rows), "seconds": round(perf_counter() - dataset_started, 3)})

event_context_started = perf_counter()
event_context_base = build_event_context(
    events,
    model_panel,
    warehouse=WAREHOUSE,
    provider=FEATURE_CONFIG.provider,
    event_families=TARGET_CONFIG.event_families,
)
event_context_base = enrich_actor_style_columns(event_context_base)
print({"step": "event_context", "rows": len(event_context_base), "seconds": round(perf_counter() - event_context_started, 3)})

congress_started = perf_counter()
congress_actor_result = build_identity_text_dataset(
    event_context_base.loc[event_context_base["event_family"].astype(str).eq("congress")].copy(),
    model_feature_metadata,
    [("actor_name", "identity__congress_actor", "task_identity_congress_actor")],
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "congress_actor_text_dataset", "rows": len(congress_actor_result.rows), "seconds": round(perf_counter() - congress_started, 3)})

analyst_started = perf_counter()
analyst_actor_result = build_identity_text_dataset(
    event_context_base.loc[event_context_base["event_family"].astype(str).isin(["analyst_rating", "price_target"])].copy(),
    model_feature_metadata,
    [("analyst_actor", "identity__analyst_actor", "task_identity_analyst_actor")],
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "analyst_actor_text_dataset", "rows": len(analyst_actor_result.rows), "seconds": round(perf_counter() - analyst_started, 3)})

congress_chamber_started = perf_counter()
congress_chamber_result = build_identity_text_dataset(
    event_context_base.loc[event_context_base["event_family"].astype(str).eq("congress")].copy(),
    model_feature_metadata,
    [("congress_chamber", "identity__congress_chamber", "task_identity_congress_chamber")],
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "congress_chamber_text_dataset", "rows": len(congress_chamber_result.rows), "seconds": round(perf_counter() - congress_chamber_started, 3)})

insider_role_started = perf_counter()
insider_role_result = build_identity_text_dataset(
    event_context_base.loc[event_context_base["event_family"].astype(str).eq("insider")].copy(),
    model_feature_metadata,
    [("insider_role", "identity__insider_role", "task_identity_insider_role")],
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "insider_role_text_dataset", "rows": len(insider_role_result.rows), "seconds": round(perf_counter() - insider_role_started, 3)})

insider_started = perf_counter()
insider_actor_result = build_identity_text_dataset(
    event_context_base.loc[event_context_base["event_family"].astype(str).eq("insider")].copy(),
    model_feature_metadata,
    [("actor_name", "identity__insider_actor", "task_identity_insider_actor")],
    config=DATASET_CONFIG,
    allowed_feature_families=ALLOWED_FEATURE_FAMILIES,
)
print({"step": "insider_actor_text_dataset", "rows": len(insider_actor_result.rows), "seconds": round(perf_counter() - insider_started, 3)})

long_frame = pd.concat(
    [
        event_result.rows,
        congress_actor_result.rows,
        analyst_actor_result.rows,
        congress_chamber_result.rows,
        insider_role_result.rows,
        insider_actor_result.rows,
    ],
    ignore_index=True,
)
long_frame["date"] = pd.to_datetime(long_frame["date"], errors="coerce").dt.normalize()
long_frame = long_frame.dropna(subset=["date", "text", "label", "task_id"]).copy()
long_frame["label"] = long_frame["label"].astype(str)

print(
    {
        "event_context_rows": len(event_context_base),
        "event_pair_rows": len(event_result.rows),
        "profile_feature_family": PROFILE_FEATURE_FAMILY,
        "feature_families": model_feature_metadata[["source", "family"]].drop_duplicates().shape[0],
        "congress_actor_rows": len(congress_actor_result.rows),
        "analyst_actor_rows": len(analyst_actor_result.rows),
        "congress_chamber_rows": len(congress_chamber_result.rows),
        "insider_role_rows": len(insider_role_result.rows),
        "insider_actor_rows": len(insider_actor_result.rows),
        "total_rows": len(long_frame),
        "tasks": long_frame["task_id"].nunique(),
    }
)

task_inventory = (
    long_frame.groupby(["target_task", "task_id"])
    .agg(rows=("label", "size"), labels=("label", "nunique"), feature_families=("feature_family", "nunique"))
    .reset_index()
    .sort_values(["target_task"])
)
display(task_inventory)
display(event_context_base[["symbol", "date", "event_family", "event_type", "country", "sector", "industry", "exchange", "actor_name", "actor_type", "analyst_actor", "congress_chamber", "insider_role"]].head(20))
display(long_frame.head(20))

## Leakage and Class-Balance Diagnostics

These checks run before training because actor-style tasks are easy to fool with leakage or sparse labels.

The notebook should make weak coverage visible rather than silently training a low-value task. Sparse actor tasks are not failures of the research idea; they are evidence that the current 1T+ universe is underpowered for that actor or role.

In [ ]:
def text_contains_label_sample(frame: pd.DataFrame, task_id: str, max_checks: int = 5000) -> pd.DataFrame:
    sample = frame.loc[frame["task_id"].eq(task_id)].head(max_checks).copy()
    if sample.empty:
        return pd.DataFrame(columns=["task_id", "checked_rows", "leaked_rows"])

    def has_exact_label_token(row: pd.Series) -> bool:
        label = re.escape(str(row["label"]).strip().lower())
        if not label:
            return False
        text = str(row["text"]).lower()
        return re.search(rf"(?<![a-z0-9_]){label}(?![a-z0-9_])", text) is not None

    leaked = sample.apply(has_exact_label_token, axis=1)
    return pd.DataFrame([{"task_id": task_id, "checked_rows": len(sample), "leaked_rows": int(leaked.sum())}])

leakage_checks = pd.concat(
    [text_contains_label_sample(long_frame, task_id) for task_id in sorted(long_frame["task_id"].unique())],
    ignore_index=True,
)
label_distribution = (
    long_frame.groupby("task_id")["label"]
    .value_counts()
    .rename("rows")
    .reset_index()
    .sort_values(["task_id", "rows"], ascending=[True, False])
)

print({"rows": len(long_frame), "tasks": long_frame["task_id"].nunique()})
display(leakage_checks)
display(label_distribution.groupby("task_id").head(20))

## Train/Test Splits

Splits are time-based to avoid training on later periods and evaluating on earlier periods.

This is appropriate for event-direction and trading-adjacent tasks because the model is evaluated on later events than it trained on. Profile context such as symbol, sector, industry, exchange, and country is treated as input feature context, not as a prediction target.

In [ ]:
def split_long_frame(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    train = frame.loc[frame["date"].le(TRAIN_END)].copy()
    dev = frame.loc[frame["date"].gt(TRAIN_END) & frame["date"].le(DEV_END)].copy()
    test = frame.loc[frame["date"].gt(DEV_END)].copy()
    if dev.empty:
        dev = test.copy()
    if test.empty:
        test = dev.copy()
    return {"train": train, "dev": dev, "test": test}


def viable_task_ids(splits: dict[str, pd.DataFrame]) -> list[str]:
    ids = []
    for task_id in sorted(splits["train"]["task_id"].unique()):
        train_labels = splits["train"].loc[splits["train"]["task_id"].eq(task_id), "label"].astype(str)
        test_labels = splits["test"].loc[splits["test"]["task_id"].eq(task_id), "label"].astype(str)
        if len(train_labels) < MIN_ROWS_PER_TASK or len(test_labels) < 2:
            continue
        if train_labels.nunique() < 2 or test_labels.nunique() < 2:
            continue
        ids.append(task_id)
    return ids

splits = split_long_frame(long_frame)
task_ids = viable_task_ids(splits)
if not task_ids:
    raise RuntimeError("No viable tasks after time split and minimum-label filters.")

splits = {name: frame.loc[frame["task_id"].isin(task_ids)].copy() for name, frame in splits.items()}
if MAX_ROWS_PER_TASK_SPLIT is not None:
    capped = {}
    for split_name, frame in splits.items():
        capped[split_name] = (
            frame.groupby("task_id", group_keys=False)
            .apply(lambda group: group.sample(n=min(len(group), MAX_ROWS_PER_TASK_SPLIT), random_state=RANDOM_SEED))
            .reset_index(drop=True)
        )
    splits = capped

split_inventory = pd.concat(
    [
        frame.groupby("task_id").agg(rows=("label", "size"), labels=("label", "nunique")).assign(split=name).reset_index()
        for name, frame in splits.items()
    ],
    ignore_index=True,
)
print({"task_ids": task_ids, "train_rows": len(splits["train"]), "dev_rows": len(splits["dev"]), "test_rows": len(splits["test"])})
display(split_inventory.pivot_table(index="task_id", columns="split", values="rows", aggfunc="sum"))
display(split_inventory.pivot_table(index="task_id", columns="split", values="labels", aggfunc="sum"))

## Build Flair DeepNCM Multitask Corpus and Model

The model uses one shared transformer embedding backbone with task-specific DeepNCM decoders. This is aligned with the research goal: event direction, actor identity, role identity, and profile context are related tasks that may teach the shared representation different aspects of smart-money behavior.

In [ ]:
MULTITASK_COLUMNS = FlairMultitaskColumns(
    text="text",
    label="label",
    label_type="label_type",
    task_id="task_id",
)

def label_dictionary_for_task(frame: pd.DataFrame, task_id: str) -> Dictionary:
    dictionary = Dictionary(add_unk=False)
    labels = frame.loc[frame["task_id"].eq(task_id), "label"].astype(str).drop_duplicates().sort_values()
    for label in labels:
        dictionary.add_item(label)
    return dictionary


def build_deepncm_multitask_model(corpus: Corpus, task_ids: list[str]) -> MultitaskModel:
    embeddings = TransformerDocumentEmbeddings(
        FLAIR_TRANSFORMER,
        fine_tune=FINE_TUNE_TRANSFORMER,
        layers="-1",
        layer_mean=False,
        allow_long_sentences=False,
        transformers_tokenizer_kwargs={"model_max_length": 512},
    )
    classifiers = []
    for task_id in task_ids:
        dictionary = label_dictionary_for_task(splits["train"], task_id)
        decoder = DeepNCMDecoder(
            label_dictionary=dictionary,
            embeddings_size=embeddings.embedding_length,
            use_encoder=DEEPNCM_USE_ENCODER,
            encoding_dim=DEEPNCM_ENCODING_DIM,
            mean_update_method=DEEPNCM_MEAN_UPDATE_METHOD,
            multi_label=False,
        )
        classifier = TextClassifier(
            embeddings,
            label_type=task_id,
            label_dictionary=dictionary,
            decoder=decoder,
            multi_label=False,
        )
        classifiers.append(classifier)
    model = MultitaskModel(
        classifiers,
        task_ids=task_ids,
        loss_factors=[1.0 for _ in task_ids],
        use_all_tasks=False,
    )
    return model.to(flair.device)


corpus_started = perf_counter()
corpus = build_multitask_corpus(splits, columns=MULTITASK_COLUMNS)
print({"step": "build_multitask_corpus", "seconds": round(perf_counter() - corpus_started, 3), "device": str(flair.device)})

model_started = perf_counter()
model = build_deepncm_multitask_model(corpus, task_ids)
print({"step": "build_deepncm_multitask_model", "seconds": round(perf_counter() - model_started, 3), "model_device": str(next(model.parameters()).device)})
print(model)

In [ ]:
trainer = ModelTrainer(model, corpus)
train_started = perf_counter()
train_result = trainer.fine_tune(
    base_path=ARTIFACT_DIR,
    learning_rate=FLAIR_LEARNING_RATE,
    mini_batch_size=FLAIR_BATCH_SIZE,
    mini_batch_chunk_size=FLAIR_BATCH_CHUNK_SIZE,
    eval_batch_size=FLAIR_EVAL_BATCH_SIZE,
    max_epochs=FLAIR_EPOCHS,
    embeddings_storage_mode="none",
    save_final_model=False,
    create_file_logs=False,
    create_loss_file=False,
)
print({"train_seconds": round(perf_counter() - train_started, 3), "train_result": train_result})

## Evaluate by Task

Evaluation is grouped by task so the notebook can distinguish three kinds of evidence:

- event-direction signal, which is closest to immediate trading behavior;
- actor/role-style signal, which tests whether high- or low-performing groups have learnable fingerprints;
- regime/context signal, which tests whether behavior clusters by issuer, sector, industry, exchange, or time period.

In [ ]:
def predict_task_classifier(model: MultitaskModel, task_id: str, rows: pd.DataFrame) -> np.ndarray:
    task = model.tasks[task_id]
    sentences = [Sentence(str(text)) for text in rows["text"].tolist()]
    task.predict(sentences, label_name="pred", mini_batch_size=FLAIR_EVAL_BATCH_SIZE)
    return np.array([sentence.get_labels("pred")[0].value for sentence in sentences])


def evaluate_multitask_model(model: MultitaskModel, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task_id, task_rows in frame.groupby("task_id", sort=True):
        y_true = task_rows["label"].astype(str).to_numpy()
        if len(np.unique(y_true)) < 2:
            rows.append(
                {
                    "task_id": task_id,
                    "target_task": task_rows["target_task"].iloc[0],
                    "rows": len(task_rows),
                    "labels": dict(task_rows["label"].astype(str).value_counts()),
                    "accuracy": np.nan,
                    "macro_f1": np.nan,
                    "macro_precision": np.nan,
                    "macro_recall": np.nan,
                    "note": "single_class_test_set",
                }
            )
            continue
        y_pred = predict_task_classifier(model, task_id, task_rows)
        rows.append(
            {
                "task_id": task_id,
                "target_task": task_rows["target_task"].iloc[0],
                "rows": len(task_rows),
                "labels": dict(task_rows["label"].astype(str).value_counts()),
                "accuracy": float(accuracy_score(y_true, y_pred)),
                "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
                "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
                "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
                "note": "ok",
            }
        )
    return pd.DataFrame(rows).sort_values(["macro_f1", "rows"], ascending=[False, False]).reset_index(drop=True)


test_results = evaluate_multitask_model(model, splits["test"])
display(test_results)

In [ ]:
analysis_lines = [
    "# Smart-Money Behavior MTL Analysis",
    "",
    f"Universe filter: FMP equities with market_cap >= {MIN_MARKET_CAP:,}.",
    f"Oracle labels were excluded; trained tasks: {', '.join(task_ids)}.",
    f"Train/dev/test rows: {len(splits['train']):,} / {len(splits['dev']):,} / {len(splits['test']):,}.",
    "",
]

if not test_results.empty:
    best = test_results.iloc[0]
    analysis_lines.append(
        f"Best task by macro F1 was {best['target_task']} with macro_f1={best['macro_f1']:.4f} over {int(best['rows']):,} test rows."
    )

    event_tasks = test_results[test_results["target_task"].astype(str).str.startswith("event_pair__")]
    if not event_tasks.empty:
        analysis_lines.append("")
        analysis_lines.append("Event-direction tasks:")
        for _, row in event_tasks.sort_values("target_task").iterrows():
            analysis_lines.append(
                f"- {row['target_task']}: accuracy={row['accuracy']:.4f}, macro_f1={row['macro_f1']:.4f}, rows={int(row['rows']):,}."
            )

    style_tasks = test_results[test_results["target_task"].astype(str).isin([
        "identity__congress_actor",
        "identity__analyst_actor",
        "identity__insider_actor",
        "identity__congress_chamber",
        "identity__insider_role",
    ])]
    if not style_tasks.empty:
        analysis_lines.append("")
        analysis_lines.append("Actor and role style tasks:")
        for _, row in style_tasks.sort_values("target_task").iterrows():
            analysis_lines.append(
                f"- {row['target_task']}: accuracy={row['accuracy']:.4f}, macro_f1={row['macro_f1']:.4f}, rows={int(row['rows']):,}."
            )

analysis_lines.extend(
    [
        "",
        "Interpretation:",
        "- Event-direction tasks test whether smart-money actions are distinguishable from issuer feature-family state on the actual event date.",
        "- Actor and role style tasks test whether actors or role groups repeatedly act in similar company states. These tasks connect prior performance differences, such as congress vs senate or CEO vs CFO/director/officer, to learnable behavior fingerprints.",
        "- Profile context is now represented as an input feature family, not as a prediction task. This should help the shared representation learn familiarity boundaries without spending capacity predicting known labels like symbol or exchange.",
        "- Actor identity learnability is not the final trading objective. It is useful if the learned fingerprint helps separate high-performing smart-money behavior from weaker or noisier behavior in downstream financial targets.",
        "- Low actor-style scores in the 1T+ universe should be read as underpowered coverage, not as proof the behavior is not learnable. The next evidence step is expanding the symbol universe and checking whether actor/role scores improve with more repeated events and realized-return metrics.",
    ]
)

analysis_text = "\n".join(analysis_lines)
print(analysis_text)

## Run Analysis

After rerunning the notebook, use the generated `Smart-Money Behavior MTL Analysis` output above as the run-specific analysis.

The important design change in this version is that profile context is no longer trained as identity targets. `symbol`, `sector`, `industry`, `exchange`, and similar context are known from the transaction and should mostly help the model learn familiarity boundaries. The notebook now includes issuer context such as country, sector, industry, and exchange as an `fmp_equity_profile` feature family.

Read the results around three questions:

- Event direction: do issuer feature families plus profile context help distinguish buy/sell, upgrade/downgrade, or raise/cut behavior on the actual event date?
- Actor and role style: do individual actors, analyst firms, congress chamber, or insider roles have learnable behavior fingerprints?
- Familiarity: does adding profile context help actor/style tasks avoid treating a familiar setup outside the actor's observed domain as equally reliable?

The 1T+ configuration remains a smoke-scale run. The next useful comparison is an ablation: actor/event tasks with and without `fmp_equity_profile`, evaluated against financial outcomes rather than just classification metrics.